# XCDM parameter-space exploration with PSO

The XCDM parametrization replaces the cosmological constant with a constant dark-energy equation-of-state parameter $w$. This notebook compares flat $\Lambda$CDM, flat XCDM, and non-flat XCDM using the same 31 cosmic-chronometer measurements.

The full calculation uses 200/500 particles/iterations for the $\Lambda$CDM baseline, 250/700 for flat XCDM, and 300/900 for non-flat XCDM.

In [ ]:
from pathlib import Path
import os
import sys
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from pso_cosmology import (
    load_cosmic_chronometers,
    h_flat_lcdm,
    h_nonflat_lcdm,
    h_flat_xcdm,
    h_nonflat_xcdm,
    run_pso,
    chi_square,
    model_statistics,
)

DATA_FILE = ROOT / "data" / "cosmic_chronometers_31.txt"
OUTPUT_DIR = Path(os.environ.get("PSO_OUTPUT_DIR", ROOT / "results" / "figures"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
QUICK = os.environ.get("PSO_QUICK", "0") == "1"

z, H_obs, sigma_H = load_cosmic_chronometers(DATA_FILE)
print(f"Loaded {len(z)} cosmic-chronometer measurements spanning z={z.min():.3f}–{z.max():.3f}.")
print("Quick mode:", QUICK)


In [ ]:
def objective_lcdm(theta):
    H0, omega_m = theta
    if not (50.0 <= H0 <= 100.0 and 0.1 <= omega_m <= 0.9):
        return np.inf
    return chi_square(h_flat_lcdm(z, H0, omega_m), H_obs, sigma_H)

def objective_flat_xcdm(theta):
    H0, omega_m, w = theta
    if not (50.0 <= H0 <= 100.0 and 0.01 <= omega_m <= 0.99 and -2.5 <= w <= 0.0):
        return np.inf
    omega_x = 1.0 - omega_m
    if not (0.0 <= omega_x <= 2.0):
        return np.inf
    return chi_square(h_flat_xcdm(z, H0, omega_m, w), H_obs, sigma_H)

def objective_nonflat_xcdm(theta):
    H0, omega_m, omega_k, w = theta
    if not (50.0 <= H0 <= 100.0 and 0.01 <= omega_m <= 1.0 and -0.5 <= omega_k <= 0.5 and -2.5 <= w <= 0.0):
        return np.inf
    omega_x = 1.0 - omega_m - omega_k
    if not (0.0 <= omega_x <= 2.0):
        return np.inf
    return chi_square(h_nonflat_xcdm(z, H0, omega_m, omega_k, w), H_obs, sigma_H)

if QUICK:
    cfg_l = dict(n_particles=50,n_iter=120,seed=42)
    cfg_f = dict(n_particles=65,n_iter=150,seed=43)
    cfg_n = dict(n_particles=75,n_iter=180,seed=44)
else:
    cfg_l = dict(n_particles=200,n_iter=500,seed=42)
    cfg_f = dict(n_particles=250,n_iter=700,seed=43)
    cfg_n = dict(n_particles=300,n_iter=900,seed=44)

lcdm = run_pso(objective_lcdm, [(50,100),(0.1,0.9)], **cfg_l)
flat_x = run_pso(objective_flat_xcdm, [(50,100),(0.01,0.99),(-2.5,0.0)], **cfg_f)
nonflat_x = run_pso(objective_nonflat_xcdm, [(50,100),(0.01,1.0),(-0.5,0.5),(-2.5,0.0)], **cfg_n)

H0_l, om_l = lcdm.best_position
H0_f, om_f, w_f = flat_x.best_position
H0_n, om_n, ok_n, w_n = nonflat_x.best_position
s_l = model_statistics(lcdm.best_score, len(z), 2)
s_f = model_statistics(flat_x.best_score, len(z), 3)
s_n = model_statistics(nonflat_x.best_score, len(z), 4)

print(f"Flat LCDM:    H0={H0_l:.6f}, Omega_m={om_l:.6f}, chi2={s_l.chi2:.6f}")
print(f"Flat XCDM:    H0={H0_f:.6f}, Omega_m={om_f:.6f}, w={w_f:.6f}, chi2={s_f.chi2:.6f}")
print(f"Non-flat XCDM:H0={H0_n:.6f}, Omega_m={om_n:.6f}, Omega_k={ok_n:.6f}, w={w_n:.6f}, chi2={s_n.chi2:.6f}")
print(f"Delta AIC: flat XCDM={s_f.aic-s_l.aic:.6f}, non-flat XCDM={s_n.aic-s_l.aic:.6f}")
print(f"Delta BIC: flat XCDM={s_f.bic-s_l.bic:.6f}, non-flat XCDM={s_n.bic-s_l.bic:.6f}")

In [ ]:
z_grid = np.linspace(0.0, 2.05, 500)
fig, ax = plt.subplots(figsize=(8, 5.5))
ax.errorbar(z, H_obs, yerr=sigma_H, fmt="o", capsize=3, alpha=0.75, label="Cosmic chronometers")
ax.plot(z_grid, h_flat_lcdm(z_grid,H0_l,om_l), lw=2.3, label=r"Flat $\Lambda$CDM")
ax.plot(z_grid, h_flat_xcdm(z_grid,H0_f,om_f,w_f), lw=2.3, ls="--", label="Flat XCDM")
ax.plot(z_grid, h_nonflat_xcdm(z_grid,H0_n,om_n,ok_n,w_n), lw=2.3, ls=":", label="Non-flat XCDM")
ax.set(xlabel="Redshift z", ylabel=r"$H(z)$ [km s$^{-1}$ Mpc$^{-1}$]", title="PSO best-fit expansion histories")
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "xcdm_best_fit_comparison.png", dpi=220)
plt.show()

In [ ]:
labels = ["Flat $\\Lambda$CDM", "Flat XCDM", "Non-flat XCDM"]
chi2_values = [s_l.chi2,s_f.chi2,s_n.chi2]
aic_values = [s_l.aic,s_f.aic,s_n.aic]
bic_values = [s_l.bic,s_f.bic,s_n.bic]
x=np.arange(len(labels)); width=0.24
fig, ax = plt.subplots(figsize=(9,5.5))
ax.bar(x-width,chi2_values,width,label=r"$\chi^2_{\min}$")
ax.bar(x,aic_values,width,label="AIC")
ax.bar(x+width,bic_values,width,label="BIC")
ax.set_xticks(x,labels)
ax.set_ylabel("Statistic")
ax.set_title("Model complexity versus goodness of fit")
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "xcdm_model_selection.png", dpi=220)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8,5.5))
ax.plot(lcdm.best_score_history,lw=2,label=r"Flat $\Lambda$CDM")
ax.plot(flat_x.best_score_history,lw=2,label="Flat XCDM")
ax.plot(nonflat_x.best_score_history,lw=2,label="Non-flat XCDM")
ax.set(xlabel="PSO iteration",ylabel=r"Best $\chi^2$ found so far",title="PSO convergence across cosmological models")
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "xcdm_convergence.png", dpi=220)
plt.show()